In [1]:
import pandas as pd

BRONZE


In [2]:
df_pedidos = pd.read_excel("pedidos_metalurgica_mezzalira.xlsx", dtype=str)

# cópia para editar
df_analise = df_pedidos.copy()
# df_analise

PRATA/LOG


In [3]:
log = []

def registrar(problema, mudança, linhas_alteradas, justificativa=""):
    log.append({
        "O que encontrei": problema,
        "O que fiz": mudança,
        "Linhas afetadas": int(linhas_alteradas),
        "Justificativa (se houve dúvida)": justificativa
        })
    print(f"{problema}: {linhas_alteradas} linha(s) afetada(s)")


In [ ]:
# remover linhas vazias
antes = len(df_analise)
df_analise = df_analise.dropna(how="all")
registrar("Linhas vazias", "Linhas completas sem dados foram removidas", antes - len(df_analise))

# remover pedidos duplicados
antes = len(df_analise)
df_analise = df_analise.drop_duplicates(subset="Pedido", keep="first")
registrar("Pedidos com o número duplicado", "Apenas o primeiro pedido foi mantido de acordo com a ordem do index", antes - len(df_analise), "Os 4 pedidos repetidos eram iguais em todas as colunas")


# df_analise

Linhas vazias: 3 linha(s) afetada(s)
Pedidos com o número duplicado: 4 linha(s) afetada(s)


In [ ]:
# alterações na coluna data

# alterar os meses tem texto
# dicionario com a chave sendo o mês como texto e o valor sendo o mês como número
meses_str = {"jan": "01", "fev": "02", "mar": "03", "abr": "04"}
# troca o texto pelo número
datas = df_analise["Data"].replace(meses_str, regex=True)
# conta quantas linhas mudaram
linhas_alteradas = (df_analise["Data"] != datas).sum()

registrar("Meses em texto nas datas", "Troquei mês em texto para seu número", linhas_alteradas)

# retirar os horarios
# separar os horarios das datas
horarios = datas.str.split(" ").str[1]
datas = datas.str.split(" ").str[0]
# contar quantas linhas tinham horario e foram alteradas
linhas_alteradas = (len(horarios) - horarios.isna().sum())

registrar("Datas com horários", "Removi os horários", linhas_alteradas, "Todos os horários eram 00:00:00")

# padronizar o formato da data

# tenta transformar cada formato e se nao conseguir deixa a linha vazia
formato_iso = pd.to_datetime(datas, format="%Y-%m-%d", errors="coerce") #formato padrão da tabela
formato_tracinho = pd.to_datetime(datas, format="%d-%m-%Y", errors="coerce") #formato com ordem trocada

# conta quantas linhas estao com o formato errado (vazias)
linhas_alteradas = formato_iso.isna().sum()

# atualiza as datas na tabela preenchendo as linhas vazias com o formato corrigido
df_analise["Data"] = formato_iso.fillna(formato_tracinho)

registrar("Datas com formatos diferentes", "Transformei todas para o formato (ano-mês-dia)", linhas_alteradas)

# testar se funcionou
# df_analise.info()
# df_analise

Meses em texto nas datas: 6 linha(s) afetada(s)
Datas com horários: 36 linha(s) afetada(s)
Datas com formatos diferentes: 6 linha(s) afetada(s)


In [ ]:
# padronizar nome das cidades

# ver quantas vezes cada cidade aparece
# df_analise["Cidade"].value_counts() 

# guardar uma cópia pra comparar com o atualizado
antes = df_analise["Cidade"].copy()

#dicionario com a chave indicando o nome errado e o valor com a correção
correcoes = {
    "recife": "Recife",
    "REC": "Recife"
}

# trocar as variações pelo nome certo na coluna das cidades em todas as linhas
df_analise["Cidade"] = df_analise["Cidade"].replace(correcoes)

# comparar o antes e depois para contar as mudanças
linhas_alteradas = (antes != df_analise["Cidade"]).sum()
registrar("Mesma cidade escrita com nomes diferentes", "Transformei as variações (recife e REC) para Recife", linhas_alteradas, "REC era abreviação de Recife")

# testar se funcionou
# df_analise["Cidade"].value_counts()

# df_analise

Mesma cidade escrita com nomes diferentes: 10 linha(s) afetada(s)


In [ ]:
# ver o tipo das variáveis
# df_analise.info()

# transformar os números salvos como texto
def converter_valor(texto):
    texto = texto.replace("R$", "").strip()
    texto = texto.replace(".", "").replace(",",".")
    numero = float(texto)
    return numero

# verifica se o texto tem R$ e conta as linhas que vão ser alteradas
linhas_alteradas = df_analise["Valor (R$)"].str.contains("R$", regex=False).sum()
# aplica a funcao em todas as linhas da coluna
df_analise["Valor (R$)"] = df_analise["Valor (R$)"].apply(converter_valor)

registrar("Valores salvos como texto e com R$", "Converti pra número e tirei o R$", linhas_alteradas)

# converter pedido para int
# contar os pedidos que estao com numero como string
linhas_alteradas = df_analise["Pedido"].apply(lambda pedido: isinstance(pedido, str)).sum()
# transforma de string para inteiro
df_analise["Pedido"] = df_analise["Pedido"].astype(int)

registrar("Números dos pedidos salvos como texto", "Converti para inteiro", linhas_alteradas)

# atualizar o index
df_analise = df_analise.reset_index(drop=True)

# testar se funcionou
# df_analise.info()
df_analise

Valores salvos como texto e com R$: 7 linha(s) afetada(s)
Números dos pedidos salvos como texto: 53 linha(s) afetada(s)


,Pedido,Data,Cliente,Cidade,Produto,Valor (R$)
0,1001,2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""",3260.0
1,1002,2026-01-02,Grades Caruaru,Caruaru,"Barra chata 1""",3570.0
2,1003,2026-01-06,Serralheria Capibaribe,Recife,Telha trapezoidal,1940.0
3,1004,2026-01-07,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,4910.0
4,1005,2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,4500.0
5,1006,2026-01-27,Irrigação Vale Forte,Petrolina,"Cantoneira 1/2""",1350.0
6,1007,2026-02-01,Grades Caruaru,Caruaru,Perfil U enrijecido,5360.0
7,1008,2026-02-04,Estaleiro Porto Novo,Recife,"Cantoneira 1/2""",3900.0
8,1009,2026-02-06,Serralheria Capibaribe,Recife,"Barra chata 1""",4040.0
9,1010,2026-02-07,Calhas Olinda,Olinda,"Tubo galvanizado 2""",5700.0


In [8]:
df_log = pd.DataFrame(log)
df_log.to_excel("log_de_alteracoes.xlsx", index=False)
display(df_log)

,O que encontrei,O que fiz,Linhas afetadas,Justificativa (se houve dúvida)
0,Linhas vazias,Linhas completas sem dados foram removidas,3,
1,Pedidos com o número duplicado,Apenas o primeiro pedido foi mantido de acordo...,4,Evitar erro de contabilidade com pedidos dupli...
2,Meses em texto nas datas,Troquei mês em texto para seu número,6,Padronização para o pandas reconhecer a data
3,Datas com horários,Removi os horários,36,Manipular as datas sem interferência do horári...
4,Datas com formatos diferentes,Transformei todas para o formato (ano-mês-dia),6,Padronizar as datas para não confundir o formato
5,Mesma cidade escrita com nomes diferentes,Transformei as variações (recife e REC) para R...,10,Padronizar a variável para acessar uma única v...
6,Valores salvos como texto e com R$,Converti pra número e tirei o R$,7,Transformar os valores para número para poder ...
7,Números dos pedidos salvos como texto,Converti para inteiro,53,Padronizar os tipos de dados


OURO


In [9]:
# criar tabela para ver qual cidade mais comprou e em qual mês

# criar coluna com os meses
df_base = df_analise.assign(Mês=df_analise["Data"].dt.month)

# dicionario com chave sendo o numero do mês e o valor sendo texto
meses_int = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr"}
# trocar na coluna
df_base["Mês"] = df_base["Mês"].replace(meses_int)

# ordem dos meses
ordem = ["Jan", "Fev", "Mar", "Abr"]
# aplica ordem na coluna dos meses
df_base["Mês"] = pd.Categorical(df_base["Mês"], categories=ordem, ordered=True)

# tabela faturamento por cidade e mês
faturamento_cidade_mes = df_base.pivot_table(
    index="Cidade",
    columns="Mês",
    values="Valor (R$)",
    aggfunc="sum",          
    fill_value=0,           
    margins=True,           
    margins_name="Total",
    observed=True
)

faturamento_cidade_mes

Mês,Jan,Fev,Mar,Abr,Total
Cidade,,,,,
Caruaru,3570.0,15030.0,17610.0,6930.0,43140.0
Jaboatão dos Guararapes,0.0,3920.0,22820.0,1120.0,27860.0
Olinda,0.0,10100.0,7710.0,11190.0,29000.0
Petrolina,9110.0,8770.0,1590.0,3610.0,23080.0
Recife,6850.0,16240.0,29730.0,2440.0,55260.0
Total,19530.0,54060.0,79460.0,25290.0,178340.0


EXTRA

In [10]:
# agrupar por produto
faturamento_por_produto = df_analise[["Produto", "Valor (R$)"]].groupby(["Produto"]).sum()
# adicionar quantidade de cada produto
faturamento_por_produto["Pedidos"] = df_analise["Produto"].value_counts()
# ordenar por valor mais alto
faturamento_por_produto = faturamento_por_produto.sort_values(by="Valor (R$)", ascending=False)

faturamento_por_produto 

,Valor (R$),Pedidos
Produto,,
"Tubo galvanizado 2""",35650.0,7
Telha trapezoidal,33270.0,10
Perfil U enrijecido,32430.0,8
Chapa de aço 3 mm,31670.0,11
"Cantoneira 1/2""",24890.0,9
"Barra chata 1""",20430.0,8


In [11]:
# agrupar por cidade para descobrir ticket médio
faturamento_por_cidade = df_analise[["Cidade", "Valor (R$)"]].groupby(["Cidade"]).sum()
# adicionar quantidade de vendas por cidade
faturamento_por_cidade["Vendas"] = df_analise["Cidade"].value_counts()
# adicionar o calculo do ticket médio
faturamento_por_cidade["Ticket Médio (R$)"] = round(faturamento_por_cidade["Valor (R$)"] / faturamento_por_cidade["Vendas"], 2)
# adicionar percentual no valor
faturamento_por_cidade["Participação (%)"] = (faturamento_por_cidade["Valor (R$)"] / faturamento_por_cidade["Valor (R$)"].sum() * 100).round(1)
# ordenar por VALOR mais alto
faturamento_por_cidade = faturamento_por_cidade.sort_values(by="Participação (%)", ascending=False)

faturamento_por_cidade


,Valor (R$),Vendas,Ticket Médio (R$),Participação (%)
Cidade,,,,
Recife,55260.0,16,3453.75,31.0
Caruaru,43140.0,11,3921.82,24.2
Olinda,29000.0,9,3222.22,16.3
Jaboatão dos Guararapes,27860.0,9,3095.56,15.6
Petrolina,23080.0,8,2885.00,12.9


In [12]:
# agrupar por cliente
faturamento_por_cliente = df_analise[["Cliente", "Valor (R$)"]].groupby(["Cliente"]).sum()
# adicionar quantidade de pedidos de cada cliente
faturamento_por_cliente["Pedidos"] = df_analise["Cliente"].value_counts()
# adicionar ticket medio de cada cliente
faturamento_por_cliente["Ticket Médio (R$)"] = round(faturamento_por_cliente["Valor (R$)"] / faturamento_por_cliente["Pedidos"], 2)
# ordenar por TICKET MÉDIO mais alto
faturamento_por_cliente = faturamento_por_cliente.sort_values(by="Ticket Médio (R$)", ascending=False)

faturamento_por_cliente

,Valor (R$),Pedidos,Ticket Médio (R$)
Cliente,,,
Grades Caruaru,29630.0,7,4232.86
Estaleiro Porto Novo,25080.0,6,4180.00
Estruturas Piedade,12250.0,3,4083.33
Esquadrias Recife Antigo,7920.0,2,3960.00
Calhas Olinda,28070.0,8,3508.75
Metalfer Agreste,13510.0,4,3377.50
Serralheria Capibaribe,17660.0,6,2943.33
Irrigação Vale Forte,14610.0,5,2922.00
Agro Estruturas São Francisco,8470.0,3,2823.33


In [15]:
df_pedidos.to_csv("bronze.csv", index=False)
df_analise.to_csv("analise.csv", index=False)
faturamento_cidade_mes.to_csv("faturamento_cidade_mes.csv")
faturamento_por_produto.to_csv("faturamento_por_produto.csv")
faturamento_por_cidade.to_csv("faturamento_por_cidade.csv")
faturamento_por_cliente.to_csv("faturamento_por_cliente.csv")